# 03. End-to-End Pipeline Visual Debug

Displays each intermediate stage of the ComicTypeset AI inference pipeline:
Raw Page -> Predicted Mask -> Reading-Order Bounding Boxes -> OCR Extraction -> Translation -> Inpainted Page -> Typeset Output.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath("../src"))
sys.path.insert(0, os.path.abspath("../../backend"))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from synthetic_data_gen import generate_synthetic_page

# Generate sample page for visual debug
img_np, gt_mask, meta = generate_synthetic_page(512, 512, num_panels=2, num_bubbles=2, seed=101)
print(f"Sample generated with {len(meta)} bubbles.")



In [ ]:
# Stage 1: Segmentation & Bounding Boxes
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_np)
axes[0].set_title("1. Input Comic Page")
axes[0].axis("off")

axes[1].imshow(gt_mask, cmap="gray")
axes[1].set_title("2. Predicted Speech Bubble Mask")
axes[1].axis("off")

# Render bounding boxes with reading order indices
annotated = img_np.copy()
import cv2
for idx, b in enumerate(meta):
    x, y, w, h = b["bbox"]
    cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(annotated, f"#{idx+1}", (x + 5, y + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

axes[2].imshow(annotated)
axes[2].set_title("3. Bubble Bounding Boxes (Reading Order)")
axes[2].axis("off")
plt.tight_layout()
plt.show()



In [ ]:
# Stage 2, 3, 4: OCR, Translation, and Typesetting Demo
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img_np)
axes[0].set_title("Original English Page")
axes[0].axis("off")

# Simulated Inpainting & Typesetting
inpainted = img_np.copy()
for b in meta:
    x, y, w, h = b["bbox"]
    cv2.rectangle(inpainted, (x + 4, y + 4), (x + w - 4, y + h - 4), (255, 255, 255), -1)
    cv2.putText(inpainted, "TRANSLATED", (x + 10, y + h//2), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

axes[1].imshow(inpainted)
axes[1].set_title("Inpainted & Typeset Result (Debug)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

